# Motor Exercise 6 — comparing response-shape models

Use one settled-speed dataset to compare a linear reference, a quadratic curve and a power law over the same normalised observations.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the plotting route; it is not evidence about your robot and is not a result you should expect to reproduce. When you are ready, change only the settings in **Use the example or your own data** and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV, set it to `False`, and enter the filename. This is the main cell you need to edit.

Expected CSV columns: `wheel`, `direction`, `PWM`, `trial_num`, and `settled_speed_cps`.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "motor_exercise06_response_shape.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The example contains a curved response so that the three candidates are visibly different. It does not reveal your robot's response shape.


In [ ]:
example_rows = []
for pwm in range(40, 151, 10):
    x = (pwm - 40) / (150 - 40)
    typical_speed = 170 + 760 * (0.72 * x ** 1.45 + 0.28 * x ** 2)
    for trial_num in range(1, 7):
        example_rows.append({
            "wheel": "left",
            "direction": "forward",
            "PWM": pwm,
            "trial_num": trial_num,
            "settled_speed_cps": typical_speed + rng.normal(0, 9),
        })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

This is where your uploaded CSV enters the notebook. Check the first rows before continuing: column names, units and labels should match the exercise.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Choose one case and fixed normalisation boundaries

Take these boundaries from the deadband, saturation and gain exercises. Keep them fixed while comparing models. Reserve complete PWM conditions before fitting.


In [ ]:
SELECTED_WHEEL = "left"
SELECTED_DIRECTION = "forward"
COMMAND_MIN = 40
COMMAND_MAX = 150
SPEED_MIN_CPS = 170
SPEED_MAX_CPS = 930
HELD_BACK_PWM = [70, 130]

selected = data.loc[
    (data["wheel"] == SELECTED_WHEEL)
    & (data["direction"] == SELECTED_DIRECTION)
].copy()


## 5. Plot the raw settled-speed observations


In [ ]:
sns.stripplot(
    data=selected,
    x="PWM",
    y="settled_speed_cps",
    jitter=0.16,
    alpha=0.5,
    native_scale=True,
)
plt.title("Raw motor-response observations")
plt.xlabel("Requested PWM")
plt.ylabel("Settled encoder speed (counts/s)")
plt.show()


## 6. Add normalised command and speed

Normalisation changes the scale, not the evidence. The raw PWM and speed columns remain in the table.


In [ ]:
response_summary = (
    selected.groupby("PWM", as_index=False)
    .agg(
        measured_speed_cps=("settled_speed_cps", "mean"),
        repeat_spread_cps=("settled_speed_cps", "std"),
    )
    .sort_values("PWM")
)
response_summary["x"] = (
    (response_summary["PWM"] - COMMAND_MIN)
    / (COMMAND_MAX - COMMAND_MIN)
)
response_summary["v"] = (
    (response_summary["measured_speed_cps"] - SPEED_MIN_CPS)
    / (SPEED_MAX_CPS - SPEED_MIN_CPS)
)
response_summary["split"] = np.where(
    response_summary["PWM"].isin(HELD_BACK_PWM),
    "held back",
    "fit",
)

response_summary


## 7. Fit the three candidates and report their equations

The three candidate equations use normalised command $x$ and normalised
speed $v$:

- linear reference: $\hat{v}=x$ (fixed, so it has no fitted coefficient);
- quadratic: $\hat{v}=A+Bx+Cx^2$;
- power law: $\hat{v}=x^\gamma$.

`np.polyfit(..., 2)` returns the quadratic coefficients in the order
**C, B, A**. This is the reverse of the order in which they appear in the
written equation. The power-law exponent $\gamma$ is selected from the
visible grid of candidate values.


In [ ]:
fit_rows = response_summary.loc[response_summary["split"] == "fit"].copy()

quadratic = np.polyfit(fit_rows["x"], fit_rows["v"], 2)
C, B, A = quadratic

gamma_candidates = np.linspace(0.5, 2.0, 301)
gamma_errors = [
    np.mean(np.abs(fit_rows["v"] - fit_rows["x"] ** gamma))
    for gamma in gamma_candidates
]
gamma = gamma_candidates[np.argmin(gamma_errors)]

response_summary["linear_prediction"] = response_summary["x"]
response_summary["quadratic_prediction"] = np.polyval(
    quadratic, response_summary["x"]
)
response_summary["power_prediction"] = response_summary["x"] ** gamma


### Read the equations and coefficient table

The printed equations use the same coefficient values as the candidate curves plotted in the next step. All values are dimensionless because both axes were normalised.


In [ ]:
coefficient_table = pd.DataFrame([
    {
        "model": "linear reference",
        "coefficient": "fixed scale",
        "symbol": "1",
        "value": 1.0,
        "units": "dimensionless",
        "meaning": "fixed identity v_hat = x; not fitted",
    },
    {
        "model": "quadratic",
        "coefficient": "intercept",
        "symbol": "A",
        "value": A,
        "units": "dimensionless",
        "meaning": "predicted normalised speed at x = 0",
    },
    {
        "model": "quadratic",
        "coefficient": "linear term",
        "symbol": "B",
        "value": B,
        "units": "dimensionless",
        "meaning": "multiplier of x",
    },
    {
        "model": "quadratic",
        "coefficient": "squared term",
        "symbol": "C",
        "value": C,
        "units": "dimensionless",
        "meaning": "multiplier of x squared",
    },
    {
        "model": "power law",
        "coefficient": "exponent",
        "symbol": "gamma",
        "value": gamma,
        "units": "dimensionless",
        "meaning": "controls the curvature of x raised to gamma",
    },
])

print("Linear reference: v_hat = x")
print(f"Quadratic: v_hat = {A:.3f} {B:+.3f}x {C:+.3f}x^2")
print(f"Power law: v_hat = x^{gamma:.3f}")
coefficient_table


## 8. Plot every candidate against the same observations


In [ ]:
prediction_plot = response_summary.melt(
    id_vars=["PWM", "x", "v", "split"],
    value_vars=[
        "linear_prediction",
        "quadratic_prediction",
        "power_prediction",
    ],
    var_name="model",
    value_name="predicted_v",
)

sns.scatterplot(
    data=response_summary,
    x="x",
    y="v",
    style="split",
    s=90,
    color="black",
)
sns.lineplot(
    data=prediction_plot,
    x="x",
    y="predicted_v",
    hue="model",
)
plt.title("Normalised observations and candidate response shapes")
plt.xlabel("Normalised requested PWM, x")
plt.ylabel("Normalised settled speed, v")
plt.show()


## 9. Inspect residual patterns

A residual is measured normalised speed minus predicted normalised speed. Patterns across `x` can reveal where a candidate repeatedly misses the response shape.


In [ ]:
residual_plot = prediction_plot.copy()
residual_plot["residual"] = (
    residual_plot["v"] - residual_plot["predicted_v"]
)

sns.relplot(
    data=residual_plot,
    x="x",
    y="residual",
    hue="model",
    style="split",
    kind="line",
    marker="o",
    height=4.5,
    aspect=1.5,
)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Normalised requested PWM, x")
plt.ylabel("Measured minus predicted v")
plt.title("Residual patterns for the three candidates")
plt.show()


## 10. Compare fitting and held-back errors


In [ ]:
error_rows = []
for model_column in [
    "linear_prediction",
    "quadratic_prediction",
    "power_prediction",
]:
    for split, rows in response_summary.groupby("split"):
        error_rows.append({
            "model": model_column.replace("_prediction", ""),
            "split": split,
            "mean_absolute_error": np.mean(
                np.abs(rows["v"] - rows[model_column])
            ),
        })

error_summary = pd.DataFrame(error_rows)
error_summary


## What to notice

- Where do the three predicted shapes visibly differ?
- Does the model with the smallest fitting error also predict held-back commands best?
- Is any improvement large compared with repeat-to-repeat variation?
- A fitted curve describes this tested response; it does not reveal a hidden mechanism.
